In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")


In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def popularidad_categoria(popularity):
    if popularity < 40:
        return "Baja"
    elif 40 <= popularity < 75:
        return "Media"
    else:
        return "Alta"

pop_udf = F.udf(popularidad_categoria, StringType())

In [0]:
df_tracks = spark.table(f"{catalogo}.{esquema_source}.spotify_tracks")
df_itunes = spark.table(f"{catalogo}.{esquema_source}.itunes_trends").withColumnRenamed("genre","itunes_genre")

In [0]:
df_tracks = df_tracks.dropna(how="all")\
                     .filter((col("track_id").isNotNull()) | (col("track_name").isNotNull()))

df_itunes = df_itunes.dropna(how="all")\
                     .filter((col("itunes_track_id").isNotNull()) | (col("track_name").isNotNull()))

In [0]:
df_tracks = df_tracks.withColumn("popularity_category", pop_udf("popularity"))

In [0]:
df_joined = df_tracks.alias("t").join(
    df_itunes.alias("i"), 
    lower(col("t.track_name")) == lower(col("i.track_name")), 
    "inner"
)

In [0]:
df_filtered_sorted = df_joined.filter(col("t.popularity") > 10).orderBy("t.popularity", ascending=False)

In [0]:

df_updated = df_filtered_sorted.select(
    "t.track_id", 
    "t.track_name", 
    "t.artists", 
    "t.popularity", 
    "t.duration_ms", 
    "t.explicit", 
    "t.popularity_category", 
    "i.itunes_genre", 
    "i.price_usd", 

    when((col("t.popularity") >= 80) & (col("t.danceability") >= 0.7), lit("Hit Bailable"))\
    .when(col("t.popularity") >= 80, lit("Hit Estandar"))\
    .otherwise(lit("Regular")).alias("is_hit"),
    "t.ingestion_date" 
)

In [0]:
df_updated.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.music_trends_transformed")